<a href="https://colab.research.google.com/github/JacquotQ/GDPR-compliance-with-Glass-Box/blob/main/Legalbert_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

!pip install torch==2.1.0+cu118 torchvision==0.16.0+cu118 --extra-index-url https://download.pytorch.org/whl/cu118
!pip install transformers==4.28.0 datasets==2.16.1
!pip install scikit-learn==1.2.2 seaborn==0.12.2 accelerate==0.24.1
!pip install numpy==1.25.2 pandas==2.0.3
!pip install evaluate



Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu118


In [2]:
from google.colab import drive

In [3]:

drive.mount('/content/drive')

Mounted at /content/drive


# violation result

In [4]:
import os
import time
import json
import warnings
import logging
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from sklearn.dummy import DummyClassifier

warnings.filterwarnings('ignore')

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger('MajorityClassBaseline')

class MajorityClassBaseline:
    def __init__(self, output_dir="./majority_baseline_results"):
        self.output_dir = output_dir
        self.label_encoder = None
        self.kf = None
        self.fold_datasets = []

        # External test data attributes
        self.X_test_external_original = None
        self.y_test_external = None

        # Model storage
        self.majority_baseline_last_fold = None

        if not os.path.exists(output_dir):
            os.makedirs(output_dir)

    def _features_to_text(self, df):
        """Convert DataFrame features to text representation"""
        texts = []
        for _, row in df.iterrows():
            text = ""
            for col, val in row.items():
                if col == 'gdpr_clause' and isinstance(val, str):
                    clauses = [clause.strip() for clause in str(val).split(',')]
                    clause_text = " and ".join(clauses)
                    text += f"GDPR clauses are {clause_text}. "
                elif col == 'Date' and isinstance(val, str):
                    text += f"Date is {val}. "
                elif col in ['country', 'company_industry'] and isinstance(val, str):
                    text += f"{col} is {val}. "
                elif isinstance(val, (int, float)):
                    if val == 1:
                        feature_name = col.replace('_', ' ').lower()
                        text += f"{feature_name} is true. "
                elif isinstance(val, str):
                    text += f"{col} is {val}. "
            texts.append(text)
        return texts

    def prepare_data(self, df, target_columns, n_splits=5):
        """Prepare data for K-fold cross validation"""
        logger.info(f"Preparing dataset for {n_splits}-fold cross validation, target columns: {target_columns}")

        exclude_columns = ['fine_amount']

        if 'gdpr_clause' in df.columns:
            exclude_columns.append('gdpr_clause')

        df_copy = df.drop(columns=exclude_columns, errors='ignore')
        logger.info(f"Excluded columns: {exclude_columns}")

        if not target_columns:
            raise ValueError("Target columns not specified.")

        missing_cols = [col for col in target_columns if col not in df_copy.columns]
        if missing_cols:
            raise ValueError(f"Missing target columns: {missing_cols}")

        # Fill missing values
        for col in df_copy.select_dtypes(include=['object']).columns:
            df_copy[col] = df_copy[col].fillna('')
        for col in df_copy.select_dtypes(include=['number']).columns:
            if col not in target_columns:
                df_copy[col] = df_copy[col].fillna(df_copy[col].median())

        target_col = target_columns[0]

        # Encode target if categorical
        if df_copy[target_col].dtype == 'object':
            logger.info(f"Target column '{target_col}' is categorical, converting to numeric")
            le = LabelEncoder()
            df_copy[target_col] = le.fit_transform(df_copy[target_col])
            self.label_encoder = le
            logger.info(f"Category mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}")

        X = df_copy.drop(columns=target_columns)
        Y = df_copy[target_col].astype('int64')

        logger.info(f"Feature count: {X.shape[1]}, Sample count: {X.shape[0]}")
        logger.info(f"Category count: {len(Y.unique())}, Category distribution: {Y.value_counts().to_dict()}")

        # Convert features to text
        X_text = self._features_to_text(X)

        # Create K-fold splits
        self.kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
        self.fold_datasets = []

        for train_idx, val_idx in self.kf.split(X_text):
            X_train_fold_text = [X_text[i] for i in train_idx]
            X_val_fold_text = [X_text[i] for i in val_idx]
            y_train_fold = Y.iloc[train_idx].values
            y_val_fold = Y.iloc[val_idx].values

            self.fold_datasets.append({
                'X_train': X_train_fold_text,
                'X_val': X_val_fold_text,
                'y_train': y_train_fold,
                'y_val': y_val_fold
            })

        logger.info(f"Created {n_splits} folds for cross-validation")
        return True

    def load_external_test_data(self, file_path, target_columns):
        """Load external test dataset"""
        logger.info(f"Loading external test data from {file_path}")

        if file_path.endswith('.csv'):
            test_df = pd.read_csv(file_path, sep=';')
        else:
            raise ValueError("Unsupported file format")

        # Handle special columns
        if 'Affected_data_volume' in test_df.columns:
            if test_df['Affected_data_volume'].dtype == 'object':
                test_df['Affected_data_volume'] = pd.to_numeric(
                    test_df['Affected_data_volume'].replace('unspecific', 0),
                    errors='coerce'
                ).fillna(0)

        # Fill missing values
        for col in test_df.select_dtypes(include=['object']).columns:
            test_df[col] = test_df[col].fillna('')
        for col in test_df.select_dtypes(include=['number']).columns:
            if col not in target_columns:
                test_df[col] = test_df[col].fillna(test_df[col].median())

        target_col = target_columns[0]
        if target_col in test_df.columns:
            if hasattr(self, 'label_encoder') and self.label_encoder and test_df[target_col].dtype == 'object':
                test_df[target_col] = test_df[target_col].fillna(self.label_encoder.classes_[0])
                unknown_categories = set(test_df[target_col].unique()) - set(self.label_encoder.classes_)
                if unknown_categories:
                    logger.warning(f"Unknown categories in test set: {unknown_categories}")
                    mode_category = self.label_encoder.classes_[0]
                    for cat in unknown_categories:
                        test_df.loc[test_df[target_col] == cat, target_col] = mode_category
                test_df[target_col] = self.label_encoder.transform(test_df[target_col])

            self.y_test_external = test_df[target_col].astype('int64').values
        else:
            self.y_test_external = None

        X_test_df = test_df.drop(columns=[target_col] if target_col in test_df.columns else [], errors='ignore')
        self.X_test_external_original = self._features_to_text(X_test_df)

        logger.info(f"External test size: {len(self.X_test_external_original)}")
        return True

    def _print_detailed_fold_metrics(self, y_true, y_pred, fold_num, model_type="Majority-Class Baseline"):
        """Print detailed metrics for each fold including per-class performance"""
        # Overall accuracy
        overall_acc = accuracy_score(y_true, y_pred)

        # Get unique classes
        unique_classes = np.unique(np.concatenate([y_true, y_pred]))

        # Calculate per-class metrics
        precision, recall, f1, support = precision_recall_fscore_support(
            y_true, y_pred, average=None, zero_division=0
        )

        # Macro averages
        macro_precision = np.mean(precision)
        macro_recall = np.mean(recall)
        macro_f1 = np.mean(f1)

        print(f"\n{'='*60}")
        print(f"{model_type} - Fold {fold_num} Detailed Metrics")
        print(f"{'='*60}")
        print(f"Overall Accuracy: {overall_acc:.4f}")
        print(f"Macro-averaged Precision: {macro_precision:.4f}")
        print(f"Macro-averaged Recall: {macro_recall:.4f}")
        print(f"Macro-averaged F1-Score: {macro_f1:.4f}")

        print(f"\nPer-Class Performance:")
        print(f"{'Class':<8} {'Precision':<10} {'Recall':<8} {'F1-Score':<10} {'Support':<8} {'Class Name'}")
        print(f"{'-'*60}")

        for i, class_idx in enumerate(unique_classes):
            if i < len(precision):
                class_name = ""
                if hasattr(self, 'label_encoder') and self.label_encoder is not None:
                    try:
                        if class_idx < len(self.label_encoder.classes_):
                            class_name = self.label_encoder.classes_[class_idx]
                    except:
                        pass

                # For binary classification, typically 0=minority, 1=majority
                if class_idx == 0:
                    class_name += " (Minority)" if class_name else "Class 0 (Minority)"
                elif class_idx == 1:
                    class_name += " (Majority)" if class_name else "Class 1 (Majority)"

                print(f"{class_idx:<8} {precision[i]:<10.4f} {recall[i]:<8.4f} {f1[i]:<10.4f} {support[i]:<8} {class_name}")

        # Class distribution in validation set
        print(f"\nClass Distribution in Validation Set:")
        unique, counts = np.unique(y_true, return_counts=True)
        for class_idx, count in zip(unique, counts):
            percentage = (count / len(y_true)) * 100
            print(f"Class {class_idx}: {count} samples ({percentage:.2f}%)")

        # Show what the majority class baseline actually predicts
        predicted_class = np.bincount(y_pred).argmax()
        print(f"\nMajority-Class Baseline Strategy:")
        print(f"Always predicts class: {predicted_class}")

        print(f"{'='*60}\n")

        return overall_acc, macro_precision, macro_recall, macro_f1

    def train_and_evaluate_majority_baseline_kfold(self):
        """Train and evaluate majority-class baseline using K-fold cross validation"""
        logger.info("Starting K-fold cross validation for Majority-Class Baseline")

        if not self.fold_datasets:
            logger.error("Fold datasets not prepared. Run prepare_data first.")
            return [], {}, []

        fold_results = []
        fold_accuracies = []
        all_train_times = []

        n_folds = len(self.fold_datasets)
        logger.info(f"Majority-class baseline K-fold will run for {n_folds} folds.")

        for fold, fold_data in enumerate(self.fold_datasets):
            logger.info(f"Training majority-class baseline fold {fold+1}/{n_folds}")

            X_train = fold_data['X_train']
            y_train = fold_data['y_train']
            X_val = fold_data['X_val']
            y_val = fold_data['y_val']

            # Show training set class distribution for this fold
            unique_train, counts_train = np.unique(y_train, return_counts=True)
            train_distribution = dict(zip(unique_train, counts_train))
            majority_class = unique_train[np.argmax(counts_train)]
            majority_percentage = (np.max(counts_train) / len(y_train)) * 100

            logger.info(f"Fold {fold+1} training set distribution: {train_distribution}")
            logger.info(f"Majority class: {majority_class} ({majority_percentage:.2f}%)")

            # Create majority-class baseline (always predicts most frequent class)
            majority_baseline = DummyClassifier(strategy='most_frequent', random_state=42)

            start_time = time.time()
            majority_baseline.fit(X_train, y_train)
            train_time = time.time() - start_time
            all_train_times.append(train_time)

            # Make predictions
            preds_val = majority_baseline.predict(X_val)

            # Calculate metrics
            acc_val = accuracy_score(y_val, preds_val)
            precision_val, recall_val, f1_val, _ = precision_recall_fscore_support(
                y_val, preds_val, average='macro', zero_division=0
            )

            # Print detailed metrics for this fold
            self._print_detailed_fold_metrics(y_val, preds_val, fold+1, "Majority-Class Baseline")

            fold_accuracies.append(acc_val)
            eval_results = {
                'eval_accuracy': acc_val,
                'eval_f1': f1_val,
                'eval_precision': precision_val,
                'eval_recall': recall_val,
                'majority_class': int(majority_class),
                'majority_percentage': majority_percentage
            }

            logger.info(f"Majority-Class Baseline Fold {fold+1} training time: {train_time:.4f}s, Accuracy: {acc_val:.4f}")

            # Save results
            fold_output_dir = os.path.join(self.output_dir, f"majority_baseline_fold_{fold+1}")
            os.makedirs(fold_output_dir, exist_ok=True)
            with open(os.path.join(fold_output_dir, 'eval_results.json'), 'w') as f:
                json.dump(eval_results, f)

            fold_results.append({
                'fold': fold+1,
                'eval_results': eval_results,
                'training_time': train_time,
                'status': 'success'
            })

            # Save the last fold model
            if fold == n_folds - 1:
                self.majority_baseline_last_fold = majority_baseline

        # Calculate averages
        avg_accuracy = np.mean([acc for acc in fold_accuracies if isinstance(acc, float)]) if fold_accuracies else 0.0

        avg_results = {
            'avg_accuracy': avg_accuracy,
            'avg_f1': np.mean([res['eval_results'].get('eval_f1', 0.0) for res in fold_results if res.get('status') == 'success']),
            'avg_precision': np.mean([res['eval_results'].get('eval_precision', 0.0) for res in fold_results if res.get('status') == 'success']),
            'avg_recall': np.mean([res['eval_results'].get('eval_recall', 0.0) for res in fold_results if res.get('status') == 'success']),
            'avg_training_time': np.mean(all_train_times) if all_train_times else 0.0
        }

        # Save average results
        with open(os.path.join(self.output_dir, 'majority_baseline_avg_results.json'), 'w') as f:
            json.dump(avg_results, f)

        logger.info(f"Majority-Class Baseline K-fold cross validation complete. Average Accuracy: {avg_accuracy:.4f}")
        return fold_results, avg_results, fold_accuracies

    def evaluate_external_test(self):
        """Evaluate majority-class baseline on external test set"""
        if self.X_test_external_original is None:
            logger.error("No external test data available")
            return None

        if self.majority_baseline_last_fold is None:
            logger.error("No majority-class baseline model available")
            return None

        if self.y_test_external is None:
            logger.warning("No external test labels available")
            return None

        logger.info("Evaluating Majority-Class Baseline on external test set")

        try:
            # Make predictions
            preds = self.majority_baseline_last_fold.predict(self.X_test_external_original)

            # Calculate metrics
            acc = accuracy_score(self.y_test_external, preds)
            precision, recall, f1, _ = precision_recall_fscore_support(
                self.y_test_external, preds, average='macro', zero_division=0
            )

            # Generate classification report
            cm = confusion_matrix(self.y_test_external, preds)
            report_dict = classification_report(
                self.y_test_external,
                preds,
                output_dict=True,
                zero_division=0,
                target_names=self.label_encoder.classes_ if hasattr(self, 'label_encoder') and self.label_encoder else None
            )
            report_df = pd.DataFrame(report_dict).transpose()
            report_df.to_csv(os.path.join(self.output_dir, 'majority_baseline_external_test_report.csv'))

            results = {
                'accuracy': acc,
                'precision': precision,
                'recall': recall,
                'f1': f1,
                'confusion_matrix': cm.tolist(),
                'classification_report': report_dict
            }

            print(f"\n{'='*60}")
            print("MAJORITY-CLASS BASELINE EXTERNAL TEST RESULTS")
            print(f"{'='*60}")
            print(f"Accuracy: {acc:.4f}")
            print(f"Precision: {precision:.4f}")
            print(f"Recall: {recall:.4f}")
            print(f"F1-Score: {f1:.4f}")
            print(f"{'='*60}")

            print(f"\nExternal Test Set Classification Report:")
            print(report_df)

            return results

        except Exception as e:
            logger.error(f"Error evaluating majority-class baseline on external test: {e}")
            return None

if __name__ == '__main__':
    # Define paths
    train_file_path = '/content/drive/MyDrive/Thesis/FINALFI.csv'
    test_file_path = '/content/drive/MyDrive/Thesis/Testdataset.csv'

    # Check if files exist
    if not os.path.exists(train_file_path):
        logger.error(f"Training file not found at {train_file_path}")
        exit()

    try:
        df = pd.read_csv(train_file_path, sep=';')
    except Exception as e:
        logger.error(f"Error reading training file: {e}")
        exit()

    target_columns = ['violation_result']

    # Initialize majority-class baseline classifier
    classifier = MajorityClassBaseline(output_dir="/content//majority_baseline_results")

    # Prepare data for K-fold cross validation
    if not classifier.prepare_data(df, target_columns, n_splits=5):
        logger.error("Data preparation failed. Exiting.")
        exit()

    # Load external test dataset if available
    try:
        if os.path.exists(test_file_path):
            classifier.load_external_test_data(test_file_path, target_columns)
        else:
            logger.warning(f"Test file not found at {test_file_path}")
    except Exception as e:
        logger.warning(f"Error loading external test data: {e}")

    # Train and evaluate Majority-Class Baseline
    logger.info("\n--- Starting Majority-Class Baseline K-Fold Cross Validation ---")
    fold_results, avg_results, fold_accuracies = classifier.train_and_evaluate_majority_baseline_kfold()

    # Print summary
    print("\n" + "="*80)
    print("MAJORITY-CLASS BASELINE K-FOLD CROSS VALIDATION SUMMARY")
    print("="*80)
    for i, acc in enumerate(fold_accuracies):
        print(f"Fold {i+1} Accuracy: {acc:.4f}")
    print("="*80)

    # Print average results
    print("\nMajority-Class Baseline - Average K-fold Results:")
    for metric, value in avg_results.items():
        print(f"{metric}: {value:.4f}")

    # Evaluate on external test set if available
    if classifier.y_test_external is not None:
        logger.info("\n--- Evaluating Majority-Class Baseline on External Test Set ---")
        external_results = classifier.evaluate_external_test()

    print(f"\nMajority-Class Baseline evaluation complete. All results saved in ./majority_baseline_results")
    print("\nNote: This baseline always predicts the most frequent class from the training set.")
    print("Any trained model should significantly outperform this baseline to be considered useful.")


Majority-Class Baseline - Fold 1 Detailed Metrics
Overall Accuracy: 0.8530
Macro-averaged Precision: 0.4265
Macro-averaged Recall: 0.5000
Macro-averaged F1-Score: 0.4603

Per-Class Performance:
Class    Precision  Recall   F1-Score   Support  Class Name
------------------------------------------------------------
0        0.0000     0.0000   0.0000     71       Class 0 (Minority)
1        0.8530     1.0000   0.9207     412      Class 1 (Majority)

Class Distribution in Validation Set:
Class 0: 71 samples (14.70%)
Class 1: 412 samples (85.30%)

Majority-Class Baseline Strategy:
Always predicts class: 1


Majority-Class Baseline - Fold 2 Detailed Metrics
Overall Accuracy: 0.8489
Macro-averaged Precision: 0.4244
Macro-averaged Recall: 0.5000
Macro-averaged F1-Score: 0.4591

Per-Class Performance:
Class    Precision  Recall   F1-Score   Support  Class Name
------------------------------------------------------------
0        0.0000     0.0000   0.0000     73       Class 0 (Minority)
1    